In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
deposit = pd.read_csv('/content/drive/MyDrive/Dataset/Preprocessed/mapping_deposit.csv')
monthly = pd.read_csv('/content/drive/MyDrive/Dataset/Preprocessed/mapping_monthly.csv')

In [ ]:
#API Key
KAKAO_API_KEY = "{KAKAO_API_KEY}"

In [ ]:
print(deposit.columns)
print(monthly.columns)

Index(['행정동명', '보증금(만원)', '임대면적(㎡)', '행정동_코드', 'medical', 'education',
       'shopping', 'bank', 'gov', 'culture', 'intercity', 'intracity', '자치구명',
       'crime_rate', 'clearance_rate', 'crime_grade'],
      dtype='object')
Index(['행정동명', '보증금(만원)', '임대료(만원)', '임대면적(㎡)', '행정동_코드', 'medical',
       'education', 'shopping', 'bank', 'gov', 'culture', 'intercity',
       'intracity', '자치구명', 'crime_rate', 'clearance_rate', 'crime_grade'],
      dtype='object')


In [ ]:
dong_list = pd.concat([
    deposit[['행정동명']],
    monthly[['행정동명']]])

dong_list = (
    dong_list
    .drop_duplicates()
    .reset_index(drop=True))

print(len(dong_list))

210


In [ ]:
import requests
import pandas as pd
import time

def address_to_coord(address):

    url = "https://dapi.kakao.com/v2/local/search/address.json"

    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": f"서울특별시 {address}"}

    response = requests.get(
        url,
        headers=headers,
        params=params)

    data = response.json()

    try:
        return (
            float(data['documents'][0]['y']),
            float(data['documents'][0]['x']))
    except:
        return None, None

In [ ]:
lat_list = []
lon_list = []

for dong in dong_list['행정동명']:

    lat, lon = address_to_coord(dong)

    lat_list.append(lat)
    lon_list.append(lon)

    time.sleep(0.1)

dong_list['lat'] = lat_list
dong_list['lon'] = lon_list

In [ ]:
dong_list[dong_list['lat'].isna()]

,행정동명,lat,lon
145,용신동,NaN,NaN


In [ ]:
dong_list.loc[
    dong_list['행정동명'] == '용신동',
    '행정동명'] = '용두동'

In [ ]:
lat, lon = address_to_coord("동대문구 용두동")

dong_list.loc[
    dong_list['행정동명'] == '용두동',
    'lat'] = lat

dong_list.loc[
    dong_list['행정동명'] == '용두동',
    'lon'] = lon

In [ ]:
dong_list[dong_list['lat'].isna()]

,행정동명,lat,lon


In [ ]:
dong_list.to_csv('/content/drive/MyDrive/Dataset/Preprocessed/dong_coord.csv', index=False)

In [ ]:
deposit['행정동명'] = deposit['행정동명'].replace(
    '용신동', '용두동')

monthly['행정동명'] = monthly['행정동명'].replace(
    '용신동', '용두동')

In [ ]:
deposit = deposit.merge(
    dong_list,
    on='행정동명',
    how='left'
)

monthly = monthly.merge(
    dong_list,
    on='행정동명',
    how='left'
)

In [ ]:
deposit[['lat','lon']].isna().sum()
monthly[['lat','lon']].isna().sum()

,0
lat,0
lon,0


In [ ]:
deposit.to_csv('/content/drive/MyDrive/Dataset/deposit.csv',index=False,)
monthly.to_csv('/content/drive/MyDrive/Dataset/monthly.csv',index=False,)